In [ ]:
from datetime import datetime
import requests
from io import BytesIO
from PIL import Image

import pandas as pd
import numpy as np
from scipy.stats import percentileofscore

import nfl_data_py as nfl

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.colors as cl
from plotly.subplots import make_subplots

from resources.plotly_theme import nfl_template
from resources.heat_map import heat_map
from resources.get_nfl_data import get_pbp_data, get_team_info, get_matchups
from resources.team_stats import get_team_stats
from resources.player_stats import get_player_stats

from resources.game_review import production_by_qtr, production_by_down, receiver_sr_down_distance, rusher_sr_down_distance, epa_box_score

pio.templates['nfl_template'] = nfl_template

In [ ]:
''' Import Data '''

# Import
team_data = get_team_info()
pbp_data = get_pbp_data(years=[2025], include_postseason=False)
pbp_data_l3y = get_pbp_data(years=[2015, 2020, 2025])

player_info = nfl.import_players()

## Data ##

# Offense
offense_stats = get_team_stats(pbp_data, unit='offense')

# Team Defense
defense_stats = get_team_stats(pbp_data, unit='defense')


print(player_info.head(2).to_string())

- Best and worst
    - Best and worst plays (EPA)

    - Best and worst performers
        - Passers, Rushers, Receivers

    - Best and worst single-game performances
        - Passers, Rushers, Receivers

- League strength
    - How did NFL compare this year to prior years?
    - "Team Stats" but for NFL. compare to prior years

- Trends (league wide)
    - Throughout year line charts
        - Under center / shotgun
        - Play action usage
        - Personnel usage

In [ ]:
''' Best and Worst EPA Plays '''

cols = ['week', 'posteam', 'posteam_score', 'defteam_score', 'defteam', 'down', 'ydstogo', 'yrdln', 'qtr', 'time', 'epa', 'desc']

top_plays = pbp_data.sort_values(by='epa', ascending=False).head(10)
worst_plays = pbp_data.sort_values(by='epa', ascending=True).head(10)

print(top_plays[cols].to_string())
print(worst_plays[cols].to_string())

fig = px.histogram(
    x=pbp_data['epa'].to_numpy(),
    template='nfl_template'
)
fig.show()

# Rushing

In [ ]:
# NOTE - No scrambles
run_data = pbp_data.loc[(pbp_data['rush'] == 1) & (pbp_data['qb_scramble'] == 0), :]

In [ ]:
''' Overall '''

by_rusher = run_data.groupby(['posteam', 'rusher']).aggregate(
    player_id=('rusher_player_id', 'first'),
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    TDs=('touchdown', 'sum'),
    Fumbles=('fumble_lost', 'sum'),
    FirstDowns=('first_down', 'sum'),
    ExplosiveRushes=('Explosive Play', lambda x: x[pbp_data['rush_attempt'] == 1].sum()),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum'),
    AvgYdsToGo=('ydstogo', 'mean'),
    AvgEP=('ep', 'mean'),
    OnScheduleAtts=('On Schedule Play', lambda x: x[run_data['rush_attempt'] == 1].sum()),
    LongDownAtts=('rush', lambda x: x[(run_data['down'] != 1) & (run_data['Distance'] == 'Long')].sum())
).sort_values(by=['Attempts'], ascending=False)

by_rusher['Yds / Att'] = round(by_rusher['Yards'] / by_rusher['Attempts'], 2)
by_rusher['Success Rate'] = round((by_rusher['Successes'] / by_rusher['Attempts']) * 100, 2)
by_rusher['EPA / Play'] = round((by_rusher['EPA'] / by_rusher['Plays']), 2)
by_rusher['1D Rate'] = round((by_rusher['FirstDowns'] / by_rusher['Attempts']) * 100, 2)
by_rusher['TD Rate'] = round((by_rusher['TDs'] / by_rusher['Attempts']) * 100, 2)
by_rusher['Explosive Rush Rate'] = round((by_rusher['ExplosiveRushes'] / by_rusher['Attempts']) * 100, 2)
by_rusher['On Schedule %'] = round((by_rusher['OnScheduleAtts'] / by_rusher['Attempts']) * 100, 2)
by_rusher['Long Down %'] = round((by_rusher['LongDownAtts'] / by_rusher['Attempts']) * 100, 2)

# by_rusher = by_rusher.loc[by_rusher['Attempts'] >= 40, :]

print(by_rusher.head().to_string())

# by_rusher['EP Perc'] = by_rusher['MedianEP'].rank(method='max', pct=True)
# by_rusher['On Schedule Perc'] = by_rusher['On Schedule %'].rank(method='max', pct=True)
# by_rusher['Situation Score'] = by_rusher[['EP Perc', 'On Schedule Perc']].mean(axis=1)

# print(by_rusher.sort_values(by='Situation Score', ascending=False).tail(10).to_string())

In [ ]:
''' Rushing Situations '''

fig = px.histogram(
    x=run_data['ep'].to_numpy(),
    title='EP',
    template='nfl_template'
)
fig.update_layout(height=350)
fig.show()
fig = px.histogram(
    x=by_rusher['On Schedule %'].to_numpy(),
    title='On Schedule %',
    template='nfl_template'
)
fig.update_layout(height=350)
fig.show()

print(f'Avg rushing situation: {run_data["ep"].mean():.2f}')
print(f'Avg On Schedule Rate (min 40 att): {by_rusher.loc[by_rusher["Attempts"] >= 40,"On Schedule %"].mean():.2f}')
print(f'Avg Long Down Share (min 40 att): {by_rusher.loc[by_rusher["Attempts"] >= 40,"Long Down %"].mean():.2f}')


print(f'Best avg rushing situation - Avg Ydstogo (min 40 att)')
print(by_rusher.loc[by_rusher['Attempts'] >= 40,:].sort_values(by='AvgYdsToGo', ascending=True).head(5).to_string())

print(f'Worst avg rushing situation - Avg Ydstogo (min 40 att)')
print(by_rusher.loc[by_rusher['Attempts'] >= 40,:].sort_values(by='AvgYdsToGo', ascending=False).head(5).to_string())

print(f'Best avg rushing situation - EP (min 40 att)')
print(by_rusher.loc[by_rusher['Attempts'] >= 40,:].sort_values(by='AvgEP', ascending=False).head(5).to_string())

print(f'Worst avg rushing situation - EP (min 40 att)')
print(by_rusher.loc[by_rusher['Attempts'] >= 40,:].sort_values(by='AvgEP', ascending=True).head(5).to_string())

print(f'Best avg rushing situation - On Schedule Rate (min 40 att)')
print(by_rusher.loc[by_rusher['Attempts'] >= 40,:].sort_values(by='On Schedule %', ascending=False).head(5).to_string())

print(f'Worst avg rushing situation - On Schedule Rate (min 40 att)')
print(by_rusher.loc[by_rusher['Attempts'] >= 40,:].sort_values(by='On Schedule %', ascending=True).head(5).to_string())

# print(by_rusher.loc[by_rusher.index.get_level_values('posteam') == 'PHI',:].to_string())

In [ ]:
for m in [40, 100]:
    print(f'Top yrds / att (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='Yds / Att', ascending=False).head(5).to_string())

    print(f'Top epa / play (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='EPA / Play', ascending=False).head(5).to_string())

    print(f'Top success rate (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='Success Rate', ascending=False).head(5).to_string())

    print(f'Top explosive rate (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='Explosive Rush Rate', ascending=False).head(5).to_string())

In [ ]:
print(f'Most carries without a touchdown')
print(by_rusher.loc[by_rusher['TDs'] == 0,:].head(3).to_string())

print(f'Most carries without an explosive rush')
print(by_rusher.loc[by_rusher['ExplosiveRushes'] == 0,:].head(3).to_string())

for m in [40, 100]:
    print(f'Worst yrds / att (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='Yds / Att', ascending=True).head(5).to_string())

    print(f'Worst epa / play (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='EPA / Play', ascending=True).head(5).to_string())

    print(f'Worst success rate (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='Success Rate', ascending=True).head(5).to_string())

    print(f'Worst explosive rate (min {m} att)')
    print(by_rusher.loc[by_rusher['Attempts'] >= m,:].sort_values(by='Explosive Rush Rate', ascending=True).head(5).to_string())

# Single-game Performances

In [ ]:
''' Rusher Performances '''

by_rusher_game = pbp_data.loc[(pbp_data['rush'] == 1),:].groupby(['game_id', 'posteam', 'rusher']).aggregate(
    player_id=('rusher_player_id', 'first'),
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    TDs=('touchdown', 'sum'),
    Fumbles=('fumble_lost', 'sum'),
    FirstDowns=('first_down', 'sum'),
    ExplosiveRushes=('Explosive Play', lambda x: x[pbp_data['rush_attempt'] == 1].sum()),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum'),
).sort_values(by=['Attempts'], ascending=False)

by_rusher_game['Yds / Att'] = round(by_rusher_game['Yards'] / by_rusher_game['Attempts'], 2)
by_rusher_game['Success Rate'] = round((by_rusher_game['Successes'] / by_rusher_game['Attempts']) * 100, 2)
by_rusher_game['EPA / Play'] = round((by_rusher_game['EPA'] / by_rusher_game['Plays']), 2)
by_rusher_game['1D Rate'] = round((by_rusher_game['FirstDowns'] / by_rusher_game['Attempts']) * 100, 2)
by_rusher_game['TD Rate'] = round((by_rusher_game['TDs'] / by_rusher_game['Attempts']) * 100, 2)

print(by_rusher_game.head(5).to_string())


In [ ]:
print(f'Top 5 Rush Yards')
print(by_rusher_game.sort_values(by='Yards', ascending=False).loc[by_rusher_game['Attempts'] > 10, :].head().to_string())

print(f'Top 5 EPA / Play (min 10 attempts)')
print(by_rusher_game.sort_values(by='EPA / Play', ascending=False).loc[by_rusher_game['Attempts'] > 10, :].head().to_string())

print(f'Top 5 Success Rate (min 10 attempts)')
print(by_rusher_game.sort_values(by='Success Rate', ascending=False).loc[by_rusher_game['Attempts'] > 10, :].head().to_string())

print(f'Most explosive rushes')
print(by_rusher_game.sort_values(by='ExplosiveRushes', ascending=False).head().to_string())

In [ ]:

print(f'Bottom 5 EPA / Play (min 10 attempts)')
print(by_rusher_game.sort_values(by='EPA / Play', ascending=True).loc[by_rusher_game['Attempts'] > 10, :].head().to_string())

print(f'Bottom 5 EPA / Play (min 20 attempts)')
print(by_rusher_game.sort_values(by='EPA / Play', ascending=True).loc[by_rusher_game['Attempts'] > 20, :].head().to_string())


# League Strength

In [ ]:
pbp_data_l3y = pbp_data_l3y.reset_index(drop=True)

In [ ]:
# Columns of interest
pass_cols = ['Pass Yards / Play', 'Pass Success Rate', 'Explosive Pass Rate', 'Pass 1D Rate', 'Completion %', 'Sack Rate', 'INT Rate']
rush_cols = ['Rush Yards / Play', 'Rush Success Rate', 'Explosive Rush Rate', 'Rush 1D Rate', 'Stuff Rate']
off_cols = ['On Schedule Rate', 'Scramble Yards / Game', 'TO Rate', 'TFL Rate', 'Penalties / Game', 'Penalty Yards / Game', 'Third Down Conv %', 'Third Down Success Rate', 'Red Zone Success Rate']
cols = pass_cols + rush_cols + off_cols

epa_cols = ['EPA / Play', 'Pass EPA / Play', 'Rush EPA / Play']

# Sort order
asc_cols = ['Stuff Rate', 'Sack Rate', 'INT Rate', 'TO Rate', 'TFL Rate', 'Penalties / Game', 'Penalty Yards / Game']
desc_cols = list(filter(lambda x: x not in asc_cols, cols))

# Display format
col_fmt = {'Pass Yards / Play': '.1f', 'Pass Success Rate': '.1%', 'Explosive Pass Rate': '.1%', 'Pass 1D Rate': '.1%', 'Completion %': '.1%', 'Sack Rate': '.1%', 'INT Rate': '.1%', 'Rush Yards / Play': '.1f', 'Rush Success Rate': '.1%', 'Explosive Rush Rate': '.1%', 'Rush 1D Rate': '.1%', 'Stuff Rate': '.1%', 'On Schedule Rate': '.1%', 'Scramble Yards / Game': '.1f', 'TO Rate': '.1%', 'TFL Rate': '.1%', 'Penalties / Game': '.1f', 'Penalty Yards / Game': '.0f', 'Third Down Conv %': '.1%', 'Third Down Success Rate': '.1%', 'Red Zone Success Rate': '.1%'}

In [ ]:
''' Team Stats (Advanced Boxscore) '''
# Game performance compared to league percentiles

cols = ['Pass Yards / Play', 'Pass Success Rate', 'Explosive Pass Rate', 'Pass 1D Rate', 'Sack Rate', 'Rush Yards / Play', 'Rush Success Rate', 'Explosive Rush Rate', 'Rush 1D Rate', 'Stuff Rate', 'Third Down Success Rate', 'Red Zone Success Rate', 'TO Rate', 'TFL Rate']

# All 2025 game offensive performances
team_stats_l3y = get_team_stats(pbp_data_l3y, unit='offense', gpby_cols=['season'])
team_stats_l3y['Red Zone Success Rate'] = team_stats_l3y['Red Zone Success Rate'].fillna(0)

print(team_stats_l3y.shape)
print(team_stats_l3y.head().to_string())

# Percentile this game performance
data = team_stats_l3y.melt(
    ignore_index=False,
    value_vars=cols,
    var_name='Metric',
    value_name='Value'
)
# )
# data['Percentile'] = 0.0

# for col in cols:
#     league_vals = all_games_team_stats[col].to_numpy()

#     order = 'asc' if col in asc_cols else 'desc'
#     for team in matchup_teams:
#         # Get team value from this game
#         conditions = (data.index == team) & (data['Metric'] == col)
#         team_val = data.loc[conditions, 'Value'].values[0]
        
#         # Percentile
#         percentile = 0
#         if order == 'asc':
#             percentile = 1 - (percentileofscore(league_vals, team_val, kind='strict') / 100)
#         else:
#             percentile = percentileofscore(league_vals, team_val, kind='weak') / 100

#         data.loc[conditions, 'Percentile'] = percentile

# data = data.merge(team_data[['team_logo_espn', 'team_color']], left_index=True, right_index=True)

data['Metric'] = pd.Categorical(data['Metric'], categories=cols, ordered=True)
data = data.sort_values(by='Metric')

# Formatting
def fmt_value(value: str, fmt: str):
    return fmt.format(value)

col_fmt_py = {col: '{0:' + fmt + '}' for col,fmt in col_fmt.items()}

data['fmt'] = data['Metric'].map(col_fmt_py)
data['value str'] = data.apply(lambda x: fmt_value(x['Value'], x['fmt']), axis=1)
# data['text'] = '<b>' + data['value str'].astype(str) + '</b> (' +  (data['Percentile']*100).round(0).astype(int).astype(str) + 'th %ile)'
data['text'] = '<b>' + data['value str'].astype(str) + '</b>'

print(data.head().to_string())

In [ ]:
seasons = data.index.unique().tolist()

# Metrics
metrics = data['Metric'].unique().tolist()
metrics = [metric.replace(' / Game', '') for metric in metrics]

# Percentiles
# away_pcts = data.loc[AWAY_TEAM, 'Percentile'].tolist()
# home_pcts = data.loc[HOME_TEAM, 'Percentile'].tolist()

# Text
# vals_23 = data.loc[2023, 'text'].tolist()
# vals_24 = data.loc[2024, 'text'].tolist()
# vals_25 = data.loc[2025, 'text'].tolist()
vals = [metrics] + [data.loc[i, 'text'].tolist() for i in seasons]
# away_vals = data.loc[AWAY_TEAM, 'text'].tolist()
# home_vals = data.loc[HOME_TEAM, 'text'].tolist()

# Color
# color_scale_len = len(px.colors.diverging.PRGn) - 1
# away_colors = [px.colors.diverging.PRGn[int(p * color_scale_len)] for p in away_pcts]
# away_text_colors = []
# for p in away_pcts:
#     if p < .15 or p > .9: away_text_colors.append('white')
#     else: away_text_colors.append('#323232')

# home_colors = [px.colors.diverging.PRGn[int(p * color_scale_len)] for p in home_pcts]
# home_text_colors = []
# for p in home_pcts:
#     if p < .15 or p > .9: home_text_colors.append('white')
#     else: home_text_colors.append('#323232')


## Figure ##

HEIGHT = 575
ROW_HEIGHT = 30
MARGIN_TOP = 60
MARGIN_BOTTOM = 20

tbl = go.Table(
    columnwidth=[4,3,3,3],
    header=dict(
        values=[''] + seasons, #['', '', ''],
        height=ROW_HEIGHT,
        fill_color='rgba(0,0,0,0)', #['#CCCCCC', SEC_COLORS[0], SEC_COLORS[1]],
        font=dict(size=18, weight=500, color='#323232'),
        line=dict(width=[0], color='#323232'),
    ),
    cells=dict(
        values=vals, #[metrics, vals_23, vals_24, vals_25],
        height=ROW_HEIGHT,
        fill_color='white', #away_colors, home_colors],
        # fill_color=['rgba(0,0,0,0)', 'white', 'white'], #away_colors, home_colors],
        font=dict(size=12, weight=['bold', 'normal', 'normal', 'normal'], color='#323232'), #away_text_colors, home_text_colors]),
        # font=dict(size=12, weight=['bold', 'normal', 'normal'], color=['#323232', '#323232', '#323232']), #away_text_colors, home_text_colors]),
        line=dict(width=0, color='#cccccc'),
        align=['left', 'center', 'center', 'center']
    ),
)

fig = go.Figure(
    data=[tbl]
)

tbl_hgt_pix = HEIGHT - MARGIN_TOP - MARGIN_BOTTOM
row_hgt = (ROW_HEIGHT / tbl_hgt_pix)
start_y = 1 - row_hgt
for l in range(len(metrics) + 1):
    y = start_y - (row_hgt * l)
    color = '#323232' if l == 0 or l == 5 or l == 10 else '#CCCCCC'
    width = 1.5 if l == 0 or l == 5 or l == 10 else 1
    fig.add_shape(
        type='line',
        yanchor='middle',
        x0=0, x1=1,
        y0=y, y1=y,
        line=dict(
            color=color,
            width=width
        )
    )

# for l in range(len(LOGOS)):
#     fig.add_layout_image(
#         source=LOGOS[l],
#         xref='paper', yref='paper',
#         xanchor='center', yanchor='bottom',
#         sizex=.09, sizey=.09,
#         x=((1/3)*(l+1)) + (1/3)/2,
#         y=.95
#     )

fig.update_layout(
    template='nfl_template',
    # title=f'<b>Offensive Team Stats</b><br><sup>Week {WEEK}: {HOME_TEAM} vs. {AWAY_TEAM}',
    width=700, height=HEIGHT,
    margin=dict(t=MARGIN_TOP,l=25,r=25,b=MARGIN_BOTTOM)
)

# Credits
fig.add_annotation(
    text=f'Percentile (in paren.) of all single-game offensive performances in 2025; <span style="color: rgba(64, 0, 75, 0.8); font-weight: bold;">purple</span> colors = lower percentiles, <span style="color: rgba(0, 68, 27, 0.8); font-weight: bold;">green</span> colors = higher percentiles<br>Figure: @clankeranalytic | Data: nfl_data_py | {datetime.today().strftime("%Y-%m-%d")}',
    showarrow=False,
    xref='paper',
    yref='paper',
    y=0, 
    x=1,
    align='right'
)

fig.show()